In [ ]:
from google.colab import files
uploaded = files.upload()

Saving asl_landmarks.npz to asl_landmarks (1).npz


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

#################################################
# LOAD DATA
#################################################

data = np.load("asl_landmarks.npz", allow_pickle=True)

X = data["X"]
y = data["y"]
classes = data["classes"]

num_classes = len(classes)

#################################################
# DATASET
#################################################

class SignDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SignDataset(X, y)

#################################################
# SPLIT
#################################################

train_size = int(0.6* len(dataset))
test_size = len(dataset) - train_size

train_ds, test_ds = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

#################################################
# MODEL
#################################################

class SignMLP(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(63, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = SignMLP(num_classes)

#################################################
# TRAINING SETUP
#################################################

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#################################################
# TRAIN LOOP
#################################################

for epoch in range(25):

    model.train()

    correct, total, loss_sum = 0, 0, 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

        preds = outputs.argmax(1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    train_acc = correct / total

    #################################################

    model.eval()

    correct, total = 0, 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:

            outputs = model(X_batch)
            preds = outputs.argmax(1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    test_acc = correct / total

    print(
        f"Epoch {epoch+1} | "
        f"Loss {loss_sum:.4f} | "
        f"Train Acc {train_acc:.3f} | "
        f"Test Acc {test_acc:.3f}"
    )

#################################################
# SAVE MODEL
#################################################

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": classes
}, "sign_mlp.pth")

print("Model saved → sign_mlp.pth")

Epoch 1 | Loss 539.5064 | Train Acc 0.713 | Test Acc 0.919
Epoch 2 | Loss 169.0568 | Train Acc 0.914 | Test Acc 0.955
Epoch 3 | Loss 121.8443 | Train Acc 0.939 | Test Acc 0.961
Epoch 4 | Loss 99.3069 | Train Acc 0.951 | Test Acc 0.966
Epoch 5 | Loss 88.0264 | Train Acc 0.958 | Test Acc 0.971
Epoch 6 | Loss 79.7537 | Train Acc 0.960 | Test Acc 0.973
Epoch 7 | Loss 70.4105 | Train Acc 0.966 | Test Acc 0.969
Epoch 8 | Loss 66.6888 | Train Acc 0.966 | Test Acc 0.971
Epoch 9 | Loss 61.5446 | Train Acc 0.970 | Test Acc 0.977
Epoch 10 | Loss 59.7894 | Train Acc 0.971 | Test Acc 0.973
Epoch 11 | Loss 54.8783 | Train Acc 0.973 | Test Acc 0.978
Epoch 12 | Loss 54.3769 | Train Acc 0.973 | Test Acc 0.974
Epoch 13 | Loss 51.5379 | Train Acc 0.973 | Test Acc 0.980
Epoch 14 | Loss 48.5499 | Train Acc 0.976 | Test Acc 0.980
Epoch 15 | Loss 47.7480 | Train Acc 0.975 | Test Acc 0.980
Epoch 16 | Loss 46.7291 | Train Acc 0.976 | Test Acc 0.979
Epoch 17 | Loss 42.6006 | Train Acc 0.978 | Test Acc 0.980
Epo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')